# 04 — Win Rate & Opportunity Funnel

**V3**: Win rate = fills with net_pnl > 0 / all filled signals.  
Rejected signals excluded from numerator AND denominator.

**V5**: Opportunity count = all evaluations (emitted + rejected).  
Capture rate = emitted / opportunity count.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
joined   = pd.read_parquet('../data/raw/joined.parquet')
filled   = pd.read_parquet('../data/raw/filled_pnl.parquet')

# V5: opportunity funnel
total_evals = len(joined)
emitted     = (joined['decision'] == 'emitted').sum()
rejected    = (joined['decision'] == 'rejected').sum()
risk_passed = joined['risk_level'].eq('pass').sum()
total_fills = len(filled)

capture_rate = emitted / total_evals if total_evals > 0 else 0

print('=== Opportunity Funnel (V5) ===')
print(f'Total evaluations  : {total_evals}')
print(f'  Rejected by engine: {rejected}')
print(f'  Emitted            : {emitted}  (capture rate: {capture_rate:.1%})')
print(f'  Risk passed        : {risk_passed}')
print(f'  Filled             : {total_fills}')

In [ ]:
# V3: win rate over filled trades only
wins = (filled['net_pnl'] > 0).sum()
win_rate = wins / len(filled) if len(filled) > 0 else float('nan')

print('=== Win Rate (V3) ===')
print(f'Fills with net_pnl > 0: {wins} / {len(filled)}')
print(f'Win rate               : {win_rate:.1%}')

In [ ]:
# Funnel bar chart
stages = ['Evaluated', 'Emitted', 'Risk Pass', 'Filled', 'Profitable']
counts = [total_evals, emitted, risk_passed, total_fills, wins]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(stages[::-1], counts[::-1], color='steelblue')
ax.set_xlabel('Count')
ax.set_title('Signal Opportunity Funnel')
for i, v in enumerate(counts[::-1]):
    ax.text(v + 0.5, i, str(v), va='center')
plt.tight_layout()
plt.savefig('../data/raw/funnel.png', dpi=120)
plt.show()

In [ ]:
# Capture efficiency: actual PnL / theoretical max PnL
from src import config as _cfg_mod
cfg = _cfg_mod.load()
cost_rate = (cfg["fee_bps"] + cfg["slippage_bps"]) / 10000

df_signals = joined.copy()
df_exec = filled.copy()

if "spread" in df_signals.columns and "expected_edge" in df_signals.columns:
    emitted_df = df_signals[df_signals["decision"] == "emitted"]
    avg_fill = df_exec["fill_size"].mean() if not df_exec.empty else 0
    theoretical = (pd.to_numeric(emitted_df["spread"], errors="coerce").abs() - cost_rate).clip(lower=0).sum() * avg_fill
    actual = pd.to_numeric(df_exec["net_pnl"], errors="coerce").sum() if not df_exec.empty else 0
    if theoretical > 0:
        efficiency = actual / theoretical
        print(f"Theoretical max PnL: ${theoretical:.4f}")
        print(f"Actual net PnL:      ${actual:.4f}")
        print(f"Capture efficiency:  {efficiency:.1%}")
        if efficiency < 0.5:
            print("WARNING: <50% capture — fill quality or latency eating the edge")
    else:
        print("No positive-edge signals emitted yet")
else:
    print("NOTE: spread or expected_edge not in joined dataset")